In [ ]:
pip install datasets

In [ ]:
!pip install -U spacy

In [ ]:
!python -m spacy init fill-config /content/base_config.cfg config.cfg

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")  # Load CoNLL-2003
train_data = dataset["train"]
test_valid_data = dataset["test"].train_test_split(test_size=0.5)
test_data = test_valid_data["test"]
valid_data = test_valid_data["train"]
print(train_data[0])  # Check first sample

In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans
def prepare_date(data,name):
  nlp = spacy.blank("en")
  db = DocBin()

  for example in data:
      doc = nlp.make_doc(" ".join(example["tokens"]))
      ents = []
      for i, tag_id in enumerate(example["ner_tags"]):
          if tag_id != 0:
              label = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"][tag_id]
              span = doc.char_span(
                  doc.text.find(example["tokens"][i]),
                  doc.text.find(example["tokens"][i]) + len(example["tokens"][i]),
                  label=label
              )
              if span is not None:
                  ents.append(span)
      doc.ents = filter_spans(ents)
      db.add(doc)

  db.to_disk(name)

In [ ]:
prepare_date(valid_data,"train.spacy")
prepare_date(valid_data,"valid.spacy")
prepare_date(test_data,"test.spacy")

In [ ]:
!python -m spacy train config.cfg \
    --output ./output \
    --paths.train ./train.spacy \
    --paths.dev ./valid.spacy \
    --verbose \
    --gpu-id 0

In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.training.example import Example

# 1. Load your trained model
nlp = spacy.load("output/model-best")

# 2. Load test data
doc_bin = DocBin().from_disk("test.spacy")
gold_docs = list(doc_bin.get_docs(nlp.vocab))

# 3. Generate predictions and prepare examples
examples = []
for gold_doc in gold_docs:
    pred_doc = nlp(gold_doc.text)
    example = Example(predicted=pred_doc, reference=gold_doc)
    examples.append(example)

# 4. Score using the nlp.evaluate() method
results = nlp.evaluate(examples)

# 5. Print results
print(f"NER F1: {results['ents_f']:.2f}")
print(f"Precision: {results['ents_p']:.2f}")
print(f"Recall: {results['ents_r']:.2f}")
print("\nPer-type scores:", results["ents_per_type"])
